In [ ]:
#引入包
import torch
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import sys
import torchvision.transforms 

#超参数定义
data_root = "./data"
img_path = "./data/img"
mask_path = "./data/mask"
batch = 20

#自定义数据类
class MyDataset(Dataset):
    def __init__(self, data_root):
        super(MyDataset, self).__init__()

        #检查路径
        if not os.path.exists(data_root):
            raise FileNotFoundError(f"数据根目录不存在：{data_root}")
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"图像文件夹不存在：{img_path}")
        if not os.path.exists(mask_path):
            raise FileNotFoundError(f"掩码文件夹不存在：{mask_path}")
        self.img_files = [
            f for f in os.listdir(img_path)
        ]
        self.mask_files = [
            f for f in os.listdir(mask_path)
        ]
    def __len__(self):
        return len(self.img_files)
    def __getitem__(self, idx):
        transforms = torchvision.transforms.ToTensor()
        img = transforms(Image.open(os.path.join(img_path, f"{idx}.png")).convert("L"))#转换单通道、张量、归一化
        mask = transforms(Image.open(os.path.join(mask_path, f"{idx}.png")).convert("L"))#转换单通道、张量、归一化
        return img

data = MyDataset(data_root)#实例化数据集

#数据迭代器
dataloader = DataLoader(dataset=data, batch_size=batch, shuffle=True, drop_last=False)

print(type(dataloader))

#残差卷积块定义
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        #主路径 Main Path
        self.mp = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3,stride=1, padding=1, bias=False)
            nn.BatchNorm2d(out_channels)
            nn.ReLU(inplace=True)
                
            nn.Conv2d(out_channels, out_channels, kernel_size=3,stride=1, padding=1, bias=False)
            nn.BatchNorm2d(out_channels)
            nn.ReLU(inplace=True)
        )
        # 捷径路径 (Shortcut / Skip Connection)
        # 如果输入输出维度不一致（通道数变了或尺寸变了），需要用 1x1 卷积调整 identity
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        identity = x
        conv = self.mp(x)
        identity = self.shortcut(identity)
        out += identity
        out = self.relu(out)
        return out

#Attention Gate Block
class AttentionGate(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        
